# Topico 3: Compressao de Dados, AEP e o Criterio de Kelly

Cenario: apostas / crescimento de patrimonio. Enunciado: `propostas/1_Simulacoes.pdf`, secao 3.

## Preparacao

In [ ]:
import sys, pathlib

REPO = pathlib.Path.cwd().parents[2]          # .../<topico>/ -> .../<proposta>/ -> raiz
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from utils.info_theory import entropy, kl_divergence, mutual_information
from utils.simulation import rng
from utils.plotting import setup_estilo, salvar_fig
import src.model as model                     # helpers especificos deste topico

setup_estilo()
# Seed 42, a mesma do script original do topico: os numeros do relatorio saem
# desta sequencia de sorteios. A ORDEM das chamadas ao gerador nos Passos 2, 8
# e 9 nao deve mudar, senao todos os numeros do relatorio mudam junto.
GEN = rng(42)
FIGURAS = pathlib.Path("figuras")

## Passo 1: Mundo real da simulacao

Definir m (ex.: 3), as probabilidades verdadeiras p_i e as cotacoes o_i.

In [ ]:
# Passo 1
RESULTADOS = ["casa", "empate", "visitante"]
M = len(RESULTADOS)
P_TRUE = np.array([0.5, 0.3, 0.2])  # probabilidades verdadeiras
ODDS = np.array([2.0, 3.0, 5.0])    # cotacoes "o para 1" da casa

H_P = entropy(P_TRUE)

# Margem da casa. Com c = 1 / sum(1/o) e precos implicitos r = c/o, vale
# W* = log2 c + D(p||r): se sum(1/o) > 1, log2 c < 0 e so ha crescimento
# positivo quando D(p||r) supera a margem.
SOMA_INV_ODDS = float((1 / ODDS).sum())
C_MERCADO = 1 / SOMA_INV_ODDS
R_IMPLICITO = C_MERCADO / ODDS
D_P_R = kl_divergence(P_TRUE, R_IMPLICITO)

# Compressao: codigo de Huffman de p contra a entropia (H <= L < H + 1)
L_HUFFMAN = model.huffman_lengths(P_TRUE)
L_MEDIO = float(P_TRUE @ L_HUFFMAN)

print(f"H(p) = {H_P:.4f} bits   (maximo para {M} simbolos: log2 {M} = {np.log2(M):.4f})")
print(f"Huffman: comprimentos {L_HUFFMAN.tolist()}, L medio = {L_MEDIO:.4f} bits, "
      f"L - H = {L_MEDIO - H_P:.4f}")
print(f"sum(1/o) = {SOMA_INV_ODDS:.4f}  -> margem da casa ~ {100 * (SOMA_INV_ODDS - 1):.1f}%")
print(f"precos implicitos r = {np.round(R_IMPLICITO, 3)}, D(p||r) = {D_P_R:.4f} bit")
print(f"W* = log2 c + D(p||r) = {np.log2(C_MERCADO):.4f} + {D_P_R:.4f} = "
      f"{np.log2(C_MERCADO) + D_P_R:.4f} bit por aposta")

## Passo 2: Temporada simulada

Gerar uma longa sequencia de n resultados iid ~ p (`sample_pmf`).

In [ ]:
# Passo 2
N_ROUNDS = 200    # rodadas por temporada
N_SEASONS = 1000  # temporadas independentes (Passo 5)

# Matriz (temporadas x rodadas) sorteada de uma vez, como sugere o Bloco 3.
RESULTADOS_MATRIZ = GEN.choice(M, size=(N_SEASONS, N_ROUNDS), p=P_TRUE)
print(f"{N_SEASONS} temporadas x {N_ROUNDS} rodadas")
print("frequencias empiricas:", np.round(np.bincount(RESULTADOS_MATRIZ.ravel(), minlength=M)
                                     / RESULTADOS_MATRIZ.size, 4))

## Passo 3: Tres estrategias

tudo no favorito (`all_in_on_favorite`), uniforme (`uniform_fractions`), Kelly (`kelly_fractions`).

In [ ]:
# Passo 3
ESTRATEGIAS = {
    "tudo no favorito": model.all_in_on_favorite(P_TRUE),
    "uniforme": model.uniform_fractions(M),
    "kelly": model.kelly_fractions(P_TRUE),
}
for nome, b in ESTRATEGIAS.items():
    print(f"{nome:17s} b = {np.round(b, 4)}")

## Passo 4: Evolucao do patrimonio

Para cada estrategia, somar em log os incrementos `model.log_growth_increments` (nao multiplicar o patrimonio).

In [ ]:
# Passo 4
# log2(patrimonio) acumulado rodada a rodada (patrimonio inicial = 1); -inf
# quando a estrategia aposta zero no resultado que saiu.
LOG_PATRIMONIO = {
    nome: np.cumsum(model.log_growth_increments(RESULTADOS_MATRIZ, b, ODDS), axis=1)
    for nome, b in ESTRATEGIAS.items()
}

# Figura 1: mediana e faixa P10-P90 das temporadas. "Tudo no favorito" fica de
# fora porque vai a -inf na primeira derrota.
rodadas = np.arange(1, N_ROUNDS + 1)
fig, ax = plt.subplots(figsize=(8, 4.5))
for nome in ["uniforme", "kelly"]:
    p10, p50, p90 = np.percentile(LOG_PATRIMONIO[nome], [10, 50, 90], axis=0)
    ax.plot(rodadas, p50, label=f"{nome} (mediana)")
    ax.fill_between(rodadas, p10, p90, alpha=0.25, label=f"{nome} (P10 a P90)")
ax.axhline(0, color="gray", linewidth=0.8)
ax.set_xlabel("rodada")
ax.set_ylabel("log2(patrimonio)")
ax.set_title("Evolucao do log2 do patrimonio")
ax.legend()
salvar_fig("t3_passo4_evolucao_patrimonio")
plt.show()

## Passo 5: Repetir a temporada

500-1000 temporadas independentes; guardar o log do patrimonio final de cada repeticao.

In [ ]:
# Passo 5
# As 1000 temporadas ja foram simuladas em paralelo no Passo 2; aqui fica so o
# log do patrimonio final de cada uma, por estrategia.
LOG_FINAL = {nome: lw[:, -1] for nome, lw in LOG_PATRIMONIO.items()}
for nome, final in LOG_FINAL.items():
    print(f"{nome:17s} mediana de log2(S_n) = {np.median(final):9.3f}")

## Passo 6: Comparar as estrategias

Taxa de crescimento tipica, media de (1/n) log2(S_n), frequencia de quase-quebra vs ganhos extraordinarios.

In [ ]:
# Passo 6
linhas = []
for nome, final in LOG_FINAL.items():
    taxa = final / N_ROUNDS  # (1/n) log2(patrimonio final)
    vivas = np.isfinite(taxa)
    linhas.append({
        "estrategia": nome,
        "arruinadas (S_n = 0)": float((~vivas).mean()),
        "taxa media": float(taxa[vivas].mean()) if vivas.any() else -np.inf,
        "desvio da taxa": float(taxa[vivas].std()) if vivas.any() else np.nan,
        "log2 S_n < -20": float((final < -20).mean()),
        "log2 S_n > 20": float((final > 20).mean()),
    })
TABELA_ESTRATEGIAS = pd.DataFrame(linhas).set_index("estrategia")
display(TABELA_ESTRATEGIAS)

# "Tudo no favorito" so sobrevive se o favorito vencer TODAS as rodadas.
P_SOBREVIVE = P_TRUE.max() ** N_ROUNDS
print(f"P(favorito vence as {N_ROUNDS} rodadas) = {P_TRUE.max()}^{N_ROUNDS} = {P_SOBREVIVE:.1e}")
print(f"fator esperado por rodada: tudo no favorito = {P_TRUE.max() * ODDS[np.argmax(P_TRUE)]:.3f}, "
      f"kelly = {float((P_TRUE * P_TRUE * ODDS).sum()):.3f}")

## Passo 7: Empirico vs teoria

Comparar a taxa empirica com W(b, p) = sum p_i log2(b_i o_i).

In [ ]:
# Passo 7
W_TEORICO = {nome: model.theoretical_growth_rate(b, P_TRUE, ODDS) for nome, b in ESTRATEGIAS.items()}
TABELA_W = TABELA_ESTRATEGIAS[["taxa media"]].copy()
TABELA_W["W(b,p) teorico"] = [W_TEORICO[nome] for nome in TABELA_W.index]
display(TABELA_W)

W_STAR = W_TEORICO["kelly"]
print(f"W* = sum p log2 o - H(p) = {float(P_TRUE @ np.log2(ODDS)):.4f} - {H_P:.4f} = {W_STAR:.4f}")

## Passo 8: AEP em acao

Histograma de (1/n) log2(S_n) para n = 10, 100, 1000; observar a concentracao em torno de W* conforme n cresce.

In [ ]:
# Passo 8
B_KELLY = ESTRATEGIAS["kelly"]

# desvio-padrao por rodada de log2(b(X) o(X)) sob p; pelo TCL, o desvio de
# (1/n) log2(patrimonio) deve ser sigma / sqrt(n)
GANHO_KELLY = np.log2(B_KELLY * ODDS)
SIGMA_KELLY = float(np.sqrt(np.sum(P_TRUE * (GANHO_KELLY - W_STAR) ** 2)))
print(f"ganhos por rodada de Kelly: {np.round(GANHO_KELLY, 3)}  (p_i o_i = {P_TRUE * ODDS})")
print(f"sigma por rodada = {SIGMA_KELLY:.4f}")

linhas_aep = []
fig, eixos = plt.subplots(1, 3, figsize=(15, 4))
for ax, n in zip(eixos, [10, 100, 1000], strict=True):
    om = GEN.choice(M, size=(N_SEASONS, n), p=P_TRUE)
    taxa = model.log_growth_increments(om, B_KELLY, ODDS).sum(axis=1) / n
    linhas_aep.append({
        "n": n,
        "media": taxa.mean(),
        "desvio empirico": taxa.std(),
        "sigma/sqrt(n) teorico": SIGMA_KELLY / np.sqrt(n),
        "fracao em W* +- 0.02": np.mean(np.abs(taxa - W_STAR) <= 0.02),
    })
    ax.hist(taxa, bins=40, density=True)
    ax.axvline(W_STAR, color="red", linestyle="--", label="W* teorico")
    ax.set_title(f"n = {n}")
    ax.set_xlabel("(1/n) log2(patrimonio final)")
    ax.legend()
salvar_fig("t3_passo8_aep_concentracao")
plt.show()

TABELA_AEP = pd.DataFrame(linhas_aep).set_index("n")
display(TABELA_AEP)

## Passo 9: Informacao lateral

Definir a conjunta resultado x dica; Kelly condicional b_i(y) = p(i | y); medir o ganho de taxa.

In [ ]:
# Passo 9
# p(y | x): a dica acerta o resultado real com probabilidade alta, mas erra.
CONFUSAO = np.array([
    [0.70, 0.20, 0.10],
    [0.15, 0.70, 0.15],
    [0.10, 0.20, 0.70],
])  # linha = resultado real x, coluna = dica y
JOINT_XY = P_TRUE[:, None] * CONFUSAO      # p(x, y) = p(x) p(y | x)
P_Y = JOINT_XY.sum(axis=0)
P_X_DADO_Y = JOINT_XY / P_Y[None, :]        # colunas normalizadas: p(x | y)

# Pares (X, Y) sorteados achatando a conjunta num vetor de 9 posicoes.
IDX = GEN.choice(JOINT_XY.size, size=(N_SEASONS, N_ROUNDS), p=JOINT_XY.flatten())
X_IDX, Y_IDX = IDX // M, IDX % M

# Kelly condicional b(y) = p(x | y) e, nos MESMOS sorteios de X, Kelly sem dica
TAXA_COM_DICA = np.log2(P_X_DADO_Y[X_IDX, Y_IDX] * ODDS[X_IDX]).mean(axis=1)
TAXA_SEM_DICA = model.log_growth_increments(X_IDX, B_KELLY, ODDS).mean(axis=1)

print(f"marginal de Y = {np.round(P_Y, 3)}")
print(f"taxa SEM dica: {TAXA_SEM_DICA.mean():.4f} bits/aposta")
print(f"taxa COM dica: {TAXA_COM_DICA.mean():.4f} bits/aposta")

## Passo 10: Ganho vs I(X; Y)

Comparar o ganho empirico do Passo 9 com I(X; Y) da conjunta definida.

In [ ]:
# Passo 10
I_XY = mutual_information(JOINT_XY)

# Ganho pareado por temporada: os dois apostadores viram os mesmos resultados,
# entao a diferenca isola o efeito da dica (sem o ruido de qual temporada saiu).
GANHO_DICA = TAXA_COM_DICA - TAXA_SEM_DICA
DELTA_W = GANHO_DICA.mean()
EP_DELTA_W = GANHO_DICA.std(ddof=1) / np.sqrt(N_SEASONS)

print(f"I(X;Y) = {I_XY:.4f} bit   (H(X|Y) = {H_P - I_XY:.4f} contra H(X) = {H_P:.4f})")
print(f"delta W empirico = {DELTA_W:.4f} bit (EP {EP_DELTA_W:.4f})")
print(f"|delta W - I| = {abs(DELTA_W - I_XY):.4f} = {abs(DELTA_W - I_XY) / EP_DELTA_W:.2f} erros-padrao")
print(f"W* + I = {W_STAR + I_XY:.4f} contra taxa com dica simulada {TAXA_COM_DICA.mean():.4f}")

## Passo 11: Crenca errada

Apostador que usa Kelly com q != p; comparar a perda de taxa com D(p||q).

In [ ]:
# Passo 11
Q_ERRADO = np.array([0.7, 0.2, 0.1])  # superestima o favorito
D_PQ = kl_divergence(P_TRUE, Q_ERRADO)

# Kelly-q contra Kelly-p nas MESMAS temporadas do Passo 2, como no Passo 10:
# por rodada, log2(p_x o_x) - log2(q_x o_x) = log2(p_x / q_x), e a cotacao some.
TAXA_KELLY_P = LOG_FINAL["kelly"] / N_ROUNDS
TAXA_KELLY_Q = model.log_growth_increments(RESULTADOS_MATRIZ, Q_ERRADO, ODDS).mean(axis=1)
PERDA = TAXA_KELLY_P - TAXA_KELLY_Q
EP_PERDA = PERDA.std(ddof=1) / np.sqrt(N_SEASONS)

print(f"taxa Kelly com p: {TAXA_KELLY_P.mean():.4f}   taxa Kelly com q: {TAXA_KELLY_Q.mean():.4f}")
print(f"perda pareada = {PERDA.mean():.4f} bit (EP {EP_PERDA:.4f})")
print(f"D(p||q) teorico = {D_PQ:.4f} bit  -> diferenca de {abs(PERDA.mean() - D_PQ) / EP_PERDA:.2f} EP")
print(f"perda / margem da casa = {D_PQ / -np.log2(C_MERCADO):.2f}")

## Cenario extra: cotacoes uniformes justas

Com a margem da casa, W* < 0 e nenhuma estrategia cresce, entao os Passos 6 e 8
nao mostram crescimento exponencial nem "ganhos extraordinarios". Aqui as mesmas
temporadas do Passo 2 sao reapostadas contra a casa ingenua do enunciado, que paga
o_i = m para todo resultado. Nesse caso W* = log2 m - H(p). Nenhum sorteio novo e
feito, entao os numeros acima nao mudam.

In [ ]:
# Cenario extra
ODDS_JUSTAS = np.full(M, float(M))
W_STAR_JUSTO = np.log2(M) - H_P

linhas_justas = []
for nome, b in ESTRATEGIAS.items():
    final = model.log_growth_increments(RESULTADOS_MATRIZ, b, ODDS_JUSTAS).sum(axis=1)
    taxa = final / N_ROUNDS
    vivas = np.isfinite(taxa)
    linhas_justas.append({
        "estrategia": nome,
        "W teorico": model.theoretical_growth_rate(b, P_TRUE, ODDS_JUSTAS),
        "taxa media": float(taxa[vivas].mean()) if vivas.any() else -np.inf,
        "log2 S_n > 20": float((final > 20).mean()),
        "mediana de S_n": float(2 ** np.median(final)),
    })
TABELA_JUSTA = pd.DataFrame(linhas_justas).set_index("estrategia")
display(TABELA_JUSTA)
print(f"W* = log2 {M} - H(p) = {np.log2(M):.4f} - {H_P:.4f} = {W_STAR_JUSTO:.4f} bit por aposta")

## Relatorio

Consolidar resultados, tabelas e figuras em `relatorio.md` (figuras salvas em `figuras/`).